# Q6: Modeling Preparation

**Phase 7:** Modeling Preparation  
**Points: 3 points**

**Focus:** Perform temporal train/test split, select features, handle
categorical variables.

**Lecture Reference:** See **Lecture 11, Notebook 3**
(`11/demo/03_pattern_analysis_modeling_prep.ipynb`), Phase 7 for
examples of temporal train/test splitting and feature preparation.
**CRITICAL:** The lecture emphasizes why temporal splitting is required
(not random split) for time series data.

In [86]:
# Q6: Modeling Preparation
# Chicago Beach Weather Sensors Dataset

import pandas as pd
import numpy as np
import os

# Create output directory if it doesn't exist
os.makedirs('output', exist_ok=True)

In [87]:
# ========================================
# STEP 1: LOAD FEATURE-ENGINEERED DATA FROM Q4
# ========================================
print("="*60)
print("Q6: MODELING PREPARATION")
print("="*60)

# Load feature-engineered data from Q4 with datetime index
datetime_col = 'Measurement Timestamp'  # Update if different

df = pd.read_csv('output/q4_features.csv')

print(f"\nLoaded {len(df):,} records with features")
print(f"Columns: {df.shape[1]}")

# Parse datetime if it exists
if datetime_col in df.columns:
    df[datetime_col] = pd.to_datetime(df[datetime_col])
    print(f"Date range: {df[datetime_col].min()} to {df[datetime_col].max()}")
else:
    print(f"Warning: Datetime column '{datetime_col}' not found")
    print(f"Available columns: {df.columns.tolist()[:5]}...")  # Show first 5


Q6: MODELING PREPARATION

Loaded 195,892 records with features
Columns: 42
Date range: 2015-04-25 09:00:00 to 2025-11-24 12:00:00


In [88]:
# ========================================
# STEP 2: SELECT TARGET VARIABLE
# ========================================
print("\n" + "="*60)
print("STEP 1: SELECTING TARGET VARIABLE")
print("="*60)

# Choose your target variable (adjust based on your analysis goals)
target = 'Air Temperature'  # Update this to your chosen target

print(f"\nTarget variable: {target}")

# Verify target exists
if target not in df.columns:
    print(f"ERROR: Target variable '{target}' not found in dataset!")
    print(f"Available columns: {df.columns.tolist()}")
    raise ValueError(f"Target variable '{target}' not found")

# Display target statistics
print(f"\nTarget variable statistics:")
print(f"  Mean: {df[target].mean():.2f}")
print(f"  Std: {df[target].std():.2f}")
print(f"  Min: {df[target].min():.2f}")
print(f"  Max: {df[target].max():.2f}")
print(f"  Missing values: {df[target].isnull().sum()}")


STEP 1: SELECTING TARGET VARIABLE

Target variable: Air Temperature

Target variable statistics:
  Mean: 12.65
  Std: 10.43
  Min: -29.78
  Max: 37.60
  Missing values: 75


In [89]:
# ========================================
# STEP 3: IDENTIFY AND EXCLUDE PROBLEMATIC FEATURES
# ========================================
print("\n" + "="*60)
print("STEP 2: FEATURE SELECTION (AVOIDING DATA LEAKAGE)")
print("="*60)

print("\nIdentifying features to EXCLUDE to prevent data leakage...")

# CRITICAL: Identify features that use the target variable
# These create data leakage - predicting the target from itself!

features_to_exclude = []

# 1. Exclude the target variable itself
features_to_exclude.append(target)
print(f"\n1. Excluding target variable: {target}")

# 2. Exclude rolling features of the target variable (DATA LEAKAGE!)
# If predicting Air Temperature, exclude air_temp_rolling_*
target_base = target.lower().replace(' ', '_')
rolling_leak_features = [col for col in df.columns 
                        if target_base in col.lower() and 'rolling' in col.lower()]
features_to_exclude.extend(rolling_leak_features)
if rolling_leak_features:
    print(f"\n2. Excluding rolling features of target (DATA LEAKAGE):")
    for feat in rolling_leak_features:
        print(f"   - {feat}")

# 3. Exclude derived features that use the target variable
# Manually identify these based on your Q4 feature engineering
if target == 'Air Temperature':
    derived_leak = ['temp_difference', 'temp_ratio', 'temp_average', 
                    'temp_category', 'comfort_index', 'temp_wind_interaction']
elif target == 'Water Temperature':
    derived_leak = ['temp_difference', 'temp_ratio', 'temp_average', 
                    'temp_category']
else:
    derived_leak = []

# Only exclude if they exist in the dataframe
derived_leak = [col for col in derived_leak if col in df.columns]
features_to_exclude.extend(derived_leak)
if derived_leak:
    print(f"\n3. Excluding derived features using target (DATA LEAKAGE):")
    for feat in derived_leak:
        print(f"   - {feat}")

# 4. Check for high correlation features (potential leakage)
print("\n4. Checking for suspiciously high correlations with target...")
numeric_cols = df.select_dtypes(include=[np.number]).columns
correlations = df[numeric_cols].corrwith(df[target]).abs().sort_values(ascending=False)
high_corr = correlations[(correlations > 0.95) & (correlations < 1.0)]
if len(high_corr) > 0:
    print("   WARNING: Features with correlation > 0.95 detected:")
    for feat, corr in high_corr.items():
        print(f"   - {feat}: {corr:.4f} (investigate for potential leakage)")
        # Don't automatically exclude - let user decide
else:
    print("   ✓ No suspiciously high correlations detected")

# Remove duplicates from exclusion list
features_to_exclude = list(set(features_to_exclude))

print(f"\nTotal features to exclude: {len(features_to_exclude)}")



STEP 2: FEATURE SELECTION (AVOIDING DATA LEAKAGE)

Identifying features to EXCLUDE to prevent data leakage...

1. Excluding target variable: Air Temperature

3. Excluding derived features using target (DATA LEAKAGE):
   - temp_category
   - comfort_index
   - temp_wind_interaction

4. Checking for suspiciously high correlations with target...
   - Wet Bulb Temperature: 0.9808 (investigate for potential leakage)

Total features to exclude: 4


In [90]:

# ========================================
# STEP 4: SELECT FEATURES
# ========================================
print("\n" + "="*60)
print("STEP 3: SELECTING VALID FEATURES")
print("="*60)

# Get all numeric columns
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
print(f"\nTotal numeric columns: {len(numeric_cols)}")

# Exclude problematic features
feature_cols = [col for col in numeric_cols if col not in features_to_exclude]
print(f"Features after exclusion: {len(feature_cols)}")

print("\nSelected features for modeling:")
for i, feat in enumerate(feature_cols, 1):
    print(f"  {i}. {feat}")


STEP 3: SELECTING VALID FEATURES

Total numeric columns: 33
Features after exclusion: 30

Selected features for modeling:
  1. Wet Bulb Temperature
  2. Humidity
  3. Rain Intensity
  4. Interval Rain
  5. Total Rain
  6. Precipitation Type
  7. Wind Direction
  8. Wind Speed
  9. Maximum Wind Speed
  10. Barometric Pressure
  11. Solar Radiation
  12. Heading
  13. Battery Life
  14. hour
  15. day_of_week
  16. month
  17. year
  18. is_weekend
  19. day_of_month
  20. quarter
  21. wind_speed_squared
  22. humidity_squared
  23. pressure_deviation
  24. wind_speed_rolling_7h
  25. wind_speed_rolling_24h
  26. wind_speed_rolling_std_7h
  27. humidity_rolling_7h
  28. humidity_rolling_24h
  29. pressure_rolling_7h
  30. pressure_rolling_24h


In [91]:
# ========================================
# STEP 5: HANDLE CATEGORICAL VARIABLES (SIMPLIFIED)
# ========================================
print("\n" + "="*60)
print("STEP 4: HANDLING CATEGORICAL VARIABLES")
print("="*60)

# Identify categorical columns
categorical_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()

# Remove datetime column if present
if datetime_col in categorical_cols:
    categorical_cols.remove(datetime_col)

print(f"\nCategorical columns found: {len(categorical_cols)}")

if categorical_cols:
    print("Categorical columns:")
    for cat in categorical_cols:
        print(f"  - {cat} ({df[cat].nunique()} unique values)")
    
    # SIMPLE FIX: Just drop categorical columns
    print("\nDropping categorical columns (not encoding)...")
    feature_cols = [col for col in feature_cols if col not in categorical_cols]
    df = df.drop(columns=categorical_cols)
    print(f"✓ Dropped {len(categorical_cols)} categorical columns")
    print(f"✓ Remaining features: {len(feature_cols)}")
else:
    print("✓ No categorical variables found")


STEP 4: HANDLING CATEGORICAL VARIABLES

Categorical columns found: 8
Categorical columns:
  - Station Name (3 unique values)
  - Measurement Timestamp Label (81318 unique values)
  - Measurement ID (195892 unique values)
  - day_name (7 unique values)
  - wind_category (4 unique values)
  - temp_category (4 unique values)
  - time_of_day (4 unique values)
  - season (4 unique values)

Dropping categorical columns (not encoding)...
✓ Dropped 8 categorical columns
✓ Remaining features: 30


In [92]:

# ========================================
# STEP 6: VERIFY NO MISSING VALUES
# ========================================
print("\n" + "="*60)
print("STEP 5: CHECKING FOR MISSING VALUES")
print("="*60)

# Check for missing values in features and target
missing_features = df[feature_cols].isnull().sum()
missing_target = df[target].isnull().sum()

print(f"\nMissing values in target ({target}): {missing_target}")

missing_any = missing_features[missing_features > 0]
if len(missing_any) > 0:
    print(f"\nFeatures with missing values: {len(missing_any)}")
    for feat, count in missing_any.items():
        pct = (count / len(df)) * 100
        print(f"  - {feat}: {count} ({pct:.2f}%)")
    
    print("\nHandling missing values...")
    # Drop rows with missing values (or use imputation if preferred)
    df = df.dropna(subset=feature_cols + [target])
    print(f"✓ Rows after dropping missing values: {len(df):,}")
else:
    print("✓ No missing values in features")




STEP 5: CHECKING FOR MISSING VALUES

Missing values in target (Air Temperature): 75

Features with missing values: 7
  - Wet Bulb Temperature: 75736 (38.66%)
  - Rain Intensity: 75736 (38.66%)
  - Total Rain: 75736 (38.66%)
  - Precipitation Type: 75736 (38.66%)
  - Barometric Pressure: 146 (0.07%)
  - Heading: 75736 (38.66%)
  - pressure_deviation: 146 (0.07%)

Handling missing values...
✓ Rows after dropping missing values: 120,081


In [93]:

# ========================================
# STEP 7: TEMPORAL TRAIN/TEST SPLIT
# ========================================
print("\n" + "="*60)
print("STEP 6: TEMPORAL TRAIN/TEST SPLIT")
print("="*60)

print("\nCRITICAL: Using TEMPORAL split (NOT random split)")
print("Why? Time series has temporal dependencies - using future data")
print("to predict the past would be data leakage!")

# CRITICAL: Sort by datetime first
if datetime_col in df.columns:
    df = df.sort_values(by=datetime_col)
    print("\n✓ Data sorted chronologically by datetime column")
    
    # Get date ranges for reporting
    train_start = df[datetime_col].iloc[0]
    train_end_idx = int(len(df) * 0.8) - 1
    train_end = df[datetime_col].iloc[train_end_idx]
    test_start = df[datetime_col].iloc[train_end_idx + 1]
    test_end = df[datetime_col].iloc[-1]
else:
    print("\n✓ Data sorted by row order (datetime column not available)")
    train_start = "N/A"
    train_end = "N/A"
    test_start = "N/A"
    test_end = "N/A"

# Define split ratio (80/20 is common)
split_ratio = 0.8
split_idx = int(len(df) * split_ratio)

print(f"\nSplit ratio: {split_ratio*100:.0f}% train / {(1-split_ratio)*100:.0f}% test")
print(f"Split index: {split_idx}")

if datetime_col in df.columns:
    print(f"\nTraining date range: {train_start} to {train_end}")
    print(f"Test date range: {test_start} to {test_end}")

    # Verify temporal split (no overlap)
    assert train_end < test_start, "ERROR: Data leakage detected! Test data overlaps with training data!"
    print("✓ Verified: No temporal overlap between train and test sets")
else:
    print("\nNote: Temporal verification skipped (datetime column not in dataset)")

# Split features (X) and target (y)
X_train = df[feature_cols].iloc[:split_idx]
X_test = df[feature_cols].iloc[split_idx:]
y_train = df[target].iloc[:split_idx]
y_test = df[target].iloc[split_idx:]

# Display split information
print(f"\nTraining set:")
print(f"  X_train shape: {X_train.shape}")
print(f"  y_train shape: {y_train.shape}")
print(f"  y_train range: [{y_train.min():.2f}, {y_train.max():.2f}]")

print(f"\nTest set:")
print(f"  X_test shape: {X_test.shape}")
print(f"  y_test shape: {y_test.shape}")
print(f"  y_test range: [{y_test.min():.2f}, {y_test.max():.2f}]")



STEP 6: TEMPORAL TRAIN/TEST SPLIT

CRITICAL: Using TEMPORAL split (NOT random split)
Why? Time series has temporal dependencies - using future data
to predict the past would be data leakage!

✓ Data sorted chronologically by datetime column

Split ratio: 80% train / 20% test
Split index: 96064

Training date range: 2015-04-25 09:00:00 to 2022-11-30 20:00:00
Test date range: 2022-11-30 21:00:00 to 2025-11-24 12:00:00
✓ Verified: No temporal overlap between train and test sets

Training set:
  X_train shape: (96064, 30)
  y_train shape: (96064,)
  y_train range: [-28.50, 37.00]

Test set:
  X_test shape: (24017, 30)
  y_test shape: (24017,)
  y_test range: [-21.40, 37.60]


In [94]:

# ========================================
# STEP 8: FINAL VERIFICATION
# ========================================
print("\n" + "="*60)
print("STEP 7: FINAL VERIFICATION")
print("="*60)

# Verify shapes match
assert X_train.shape[0] == y_train.shape[0], "Train set size mismatch!"
assert X_test.shape[0] == y_test.shape[0], "Test set size mismatch!"
assert X_train.shape[1] == X_test.shape[1], "Feature count mismatch!"
print("✓ All shapes verified")

# Verify no missing values
assert X_train.isnull().sum().sum() == 0, "Missing values in X_train!"
assert X_test.isnull().sum().sum() == 0, "Missing values in X_test!"
assert y_train.isnull().sum() == 0, "Missing values in y_train!"
assert y_test.isnull().sum() == 0, "Missing values in y_test!"
print("✓ No missing values")

# Verify no infinite values
assert not np.isinf(X_train.values).any(), "Infinite values in X_train!"
assert not np.isinf(X_test.values).any(), "Infinite values in X_test!"
assert not np.isinf(y_train.values).any(), "Infinite values in y_train!"
assert not np.isinf(y_test.values).any(), "Infinite values in y_test!"
print("✓ No infinite values")



STEP 7: FINAL VERIFICATION
✓ All shapes verified
✓ No missing values
✓ No infinite values


In [95]:

# ========================================
# SAVE ARTIFACT 1 & 2: X_train.csv, X_test.csv
# ========================================
print("\n" + "="*60)
print("SAVING ARTIFACTS")
print("="*60)

print("\nSaving training features...")
X_train.to_csv('output/q6_X_train.csv', index=False)
print(f"✓ Saved: output/q6_X_train.csv ({X_train.shape[0]} rows × {X_train.shape[1]} cols)")

print("\nSaving test features...")
X_test.to_csv('output/q6_X_test.csv', index=False)
print(f"✓ Saved: output/q6_X_test.csv ({X_test.shape[0]} rows × {X_test.shape[1]} cols)")



SAVING ARTIFACTS

Saving training features...
✓ Saved: output/q6_X_train.csv (96064 rows × 30 cols)

Saving test features...
✓ Saved: output/q6_X_test.csv (24017 rows × 30 cols)


In [96]:

# ========================================
# SAVE ARTIFACT 3 & 4: y_train.csv, y_test.csv
# ========================================
print("\nSaving training target...")
y_train_df = pd.DataFrame(y_train, columns=[target])
y_train_df.to_csv('output/q6_y_train.csv', index=False)
print(f"✓ Saved: output/q6_y_train.csv ({len(y_train_df)} rows)")

print("\nSaving test target...")
y_test_df = pd.DataFrame(y_test, columns=[target])
y_test_df.to_csv('output/q6_y_test.csv', index=False)
print(f"✓ Saved: output/q6_y_test.csv ({len(y_test_df)} rows)")



Saving training target...
✓ Saved: output/q6_y_train.csv (96064 rows)

Saving test target...
✓ Saved: output/q6_y_test.csv (24017 rows)


In [97]:

# ========================================
# SAVE ARTIFACT 5: q6_train_test_info.txt
# ========================================
print("\nSaving train/test split information...")

# Calculate total duration
train_duration = train_end - train_start
test_duration = test_end - test_start

with open('output/q6_train_test_info.txt', 'w') as f:
    f.write("TRAIN/TEST SPLIT INFORMATION\n")
    f.write("=" * 60 + "\n\n")
    
    f.write(f"Split Method: Temporal ({split_ratio*100:.0f}/{(1-split_ratio)*100:.0f} split by time)\n")
    f.write("Why temporal split? Time series has temporal dependencies.\n")
    f.write("Random split would cause data leakage.\n\n")
    
    f.write(f"Training Set Size: {len(X_train):,} samples\n")
    f.write(f"Test Set Size: {len(X_test):,} samples\n\n")
    
    f.write(f"Training Date Range: {train_start} to {train_end}\n")
    f.write(f"Training Duration: {train_duration.days} days\n\n")
    
    f.write(f"Test Date Range: {test_start} to {test_end}\n")
    f.write(f"Test Duration: {test_duration.days} days\n\n")
    
    f.write(f"Number of Features: {X_train.shape[1]}\n")
    f.write(f"Target Variable: {target}\n\n")
    
    f.write("Features Excluded (to prevent data leakage):\n")
    for feat in features_to_exclude:
        f.write(f"  - {feat}\n")
    
    f.write("\nFeatures Included:\n")
    for feat in feature_cols[:20]:  # List first 20
        f.write(f"  - {feat}\n")
    if len(feature_cols) > 20:
        f.write(f"  ... and {len(feature_cols) - 20} more features\n")

print("✓ Saved: output/q6_train_test_info.txt")



Saving train/test split information...
✓ Saved: output/q6_train_test_info.txt


In [98]:

# ========================================
# VERIFICATION
# ========================================
print("\n" + "="*60)
print("VERIFICATION")
print("="*60)

# Verify files exist
print("\nVerifying output files:")
output_files = [
    'output/q6_X_train.csv',
    'output/q6_X_test.csv',
    'output/q6_y_train.csv',
    'output/q6_y_test.csv',
    'output/q6_train_test_info.txt'
]
for file in output_files:
    if os.path.exists(file):
        size = os.path.getsize(file) / 1024  # KB
        print(f"  ✓ {file} ({size:.1f} KB)")
    else:
        print(f"  ✗ {file} NOT FOUND!")

# ========================================
# SUMMARY
# ========================================
print("\n" + "="*60)
print("Q6 COMPLETE - All artifacts created successfully!")
print("="*60)
print("\nFiles created:")
print("  1. output/q6_X_train.csv")
print("  2. output/q6_X_test.csv")
print("  3. output/q6_y_train.csv")
print("  4. output/q6_y_test.csv")
print("  5. output/q6_train_test_info.txt")
print("\nModeling Preparation Summary:")
print(f"  - Target variable: {target}")
print(f"  - Total features: {X_train.shape[1]}")
print(f"  - Features excluded (data leakage prevention): {len(features_to_exclude)}")
print(f"  - Training samples: {len(X_train):,}")
print(f"  - Test samples: {len(X_test):,}")
print(f"  - Split method: Temporal (80/20)")
print(f"  - Training period: {train_duration.days} days")
print(f"  - Test period: {test_duration.days} days")
print("\nData Leakage Prevention:")
print("  ✓ Temporal split used (not random)")
print("  ✓ Target variable excluded from features")
print("  ✓ Rolling features of target excluded")
print("  ✓ Derived features using target excluded")
print("  ✓ No temporal overlap between train/test")
print("\nNext: Proceed to Q7 for modeling")
print("="*60)


VERIFICATION

Verifying output files:
  ✓ output/q6_X_train.csv (21145.7 KB)
  ✓ output/q6_X_test.csv (5272.1 KB)
  ✓ output/q6_y_train.csv (445.5 KB)
  ✓ output/q6_y_test.csv (110.3 KB)
  ✓ output/q6_train_test_info.txt (1.0 KB)

Q6 COMPLETE - All artifacts created successfully!

Files created:
  1. output/q6_X_train.csv
  2. output/q6_X_test.csv
  3. output/q6_y_train.csv
  4. output/q6_y_test.csv
  5. output/q6_train_test_info.txt

Modeling Preparation Summary:
  - Target variable: Air Temperature
  - Total features: 30
  - Features excluded (data leakage prevention): 4
  - Training samples: 96,064
  - Test samples: 24,017
  - Split method: Temporal (80/20)
  - Training period: 2776 days
  - Test period: 1089 days

Data Leakage Prevention:
  ✓ Temporal split used (not random)
  ✓ Target variable excluded from features
  ✓ Rolling features of target excluded
  ✓ Derived features using target excluded
  ✓ No temporal overlap between train/test

Next: Proceed to Q7 for modeling
